# 3. De novo design: generate a backbone and its sequence

- **Input:** A target length of 60 residues, without a starting sequence or backbone.
- **Model information:** RFdiffusion uses learned structural patterns to generate a backbone; ProteinMPNN uses that backbone to design sequences.
- **Design objective:** Propose a new backbone and sequences compatible with it.

Generate a backbone, sample three sequences, and compare a selected sequence's predicted structure with the generated backbone.


## Setup
Select **Runtime → Change runtime type → T4 GPU**, then run the two cells below to load the tools and models.


In [ ]:
!git -C BME305-protein-design-tools pull -q 2>/dev/null || git clone -q https://github.com/RomeroLab/BME305-protein-design-tools.git
%pip -q install --upgrade ./BME305-protein-design-tools
from protein_design_tools import load_pdb, load_ESMFold, load_ESM2, load_MPNN, load_RFdiffusion, predict_structure, MPNNdesign, ESM2design, RFdiffusion, alignment, view_structure, inspect_prediction, save_results


In [ ]:
load_MPNN()
load_RFdiffusion()
load_ESMFold();


## 1. Generate a backbone
We specify only length. RFdiffusion removes noise over 50 steps to propose a backbone. Rotate the result and inspect its helices, sheets, and loops.


In [ ]:
reference_pdb = RFdiffusion(length=60, steps=50, seed=7)
view_structure(reference_pdb)


## 2. Design sequences
ProteinMPNN samples sequences conditioned on the generated backbone. The glycine placeholders in the RFdiffusion output are replaced by designed amino acids. Higher temperature usually produces more diverse sequences.


In [ ]:
designs = MPNNdesign(reference_pdb, num_sequences=3, temperature=0.1, seed=7)


## 3. Compare sequences
Compare the three designs and identify positions where they agree or differ. Sequence identity is reported relative to design 1; there is no natural reference sequence for this backbone.


In [ ]:
alignment(designs)


## 4. Predict and compare structures
Predict a selected design with ESMFold and compare it with the RFdiffusion backbone. ESMFold uses the sequence alone, without the reference coordinates. RMSD measures backbone agreement; pLDDT describes confidence in the prediction. Inspect the overall fold and regions of disagreement alongside their confidence.


In [ ]:
prediction_pdb = predict_structure(designs['design_1'], num_recycles=1)
metrics = inspect_prediction(reference_pdb, prediction_pdb)


## Discuss
- Does the predicted fold match RFdiffusion's proposed backbone? Identify one agreement and one discrepancy.
- Why do we need both a sequence designer and an independent structure prediction step after backbone generation?

## Save results
Download the designed sequences and structures for further analysis.


In [ ]:
bundle = save_results(designs, reference_pdb, prediction_pdb)
from google.colab import files
files.download(str(bundle))


## Reuse the tools
The same tools accept your own sequences and structures. See the [function reference](https://github.com/RomeroLab/BME305-protein-design-tools#function-reference) for use in later assignments.

## References
[RFdiffusion](https://github.com/RosettaCommons/RFdiffusion) · [Colab adaptation](https://github.com/sokrypton/RFdiffusion) · [ProteinMPNN](https://github.com/dauparas/ProteinMPNN) · [ESMFold](https://github.com/facebookresearch/esm)
